# 09b — Diagnósticos de trayectoria (D33/D35)

Capa 3 del rediseño del eje del TFI: solo descriptivo, sobre
`data/05_trayectoria_distancia/{pares_h1,pares_h4}` (Capa 2) ya construida en
`09_trayectorias_distancia.ipynb` -- no entrena ningún modelo.

1. Tendencia general por trimestre (mediana ponderada de la variación,
   dirección, matriz de transición).
2. Regresión a la media (variación promedio por decil de `ratio_lp_t`) --
   línea base contra la que se va a medir cualquier modelo futuro.
3. Match a `t+4` desagregado (ausencia estructural por rotación, vivienda
   que no reaparece, hogar que no supera la confirmación) y cuántos pares
   se recuperarían con una regla de confirmación menos estricta (solo se
   cuantifica, no se aplica).
4. Conteos finales de pares `t+1`/`t+4`, train/holdout (split de D24/D32).
5. Subanálisis de Laura: salida de la pobreza entre los pobres (`pobre_t`),
   mismo contenido que la notebook `09_etiqueta_salida_pobreza.ipynb`
   anterior, reescrito sobre la Capa 2 en vez de reconstruir el join a mano.

**Lee** `data/05_trayectoria_distancia/`, `data/03_panel/{pares_h1,pares_h4}`
(para `n_altas`/`n_bajas`/`cambio_composicion`, no están en la Capa 2) y
`data/02_nucleo/hogar` (para la clasificación vivienda/hogar de la sección
3, vía `cascada.clasificar_vinculacion_trimestre`). **Escribe** solo CSV en
`data/meta/`.

In [ ]:
import sys
from pathlib import Path


def _raiz_repo(desde: Path | None = None) -> Path:
    actual = (desde or Path.cwd()).resolve()
    for candidato in (actual, *actual.parents):
        if (candidato / "pytest.ini").exists():
            return candidato
    raise FileNotFoundError(f"no se encontró la raíz del repo subiendo desde {actual}")


REPO = _raiz_repo()
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import pandas as pd

import cascada
import panel
import trayectoria_distancia
from armonizacion import trimestres_publicados
from cascada import tiene_destino_valido

TRAYECTORIA = REPO / "data" / "05_trayectoria_distancia"
PANEL_DIR = REPO / "data" / "03_panel"
NUCLEO = REPO / "data" / "02_nucleo"
META = REPO / "data" / "meta"

TRIMESTRES_DISTANCIA = [t for t in trimestres_publicados() if t >= (2016, 2)]
ORIGENES_H1 = [t for t in TRIMESTRES_DISTANCIA if tiene_destino_valido(*t, 1)]
ORIGENES_H4 = [t for t in TRIMESTRES_DISTANCIA if tiene_destino_valido(*t, 4)]


def _cargar_capa2(h: int, origenes: list[tuple[int, int]]) -> pd.DataFrame:
    partes = []
    for anio, trimestre in origenes:
        p = TRAYECTORIA / f"pares_h{h}" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
        if p.exists():
            partes.append(pd.read_parquet(p))
    return pd.concat(partes, ignore_index=True)


pares_h1 = _cargar_capa2(1, ORIGENES_H1)
pares_h4 = _cargar_capa2(4, ORIGENES_H4)
print(f"pares_h1: {len(pares_h1):,} -- pares_h4: {len(pares_h4):,}")


## 1. Tendencia general por trimestre

Mediana ponderada por `PONDIH_t` de `variacion_simetrica`/`variacion_log` por
trimestre de origen (`trayectoria_distancia.mediana_ponderada`), proporción
de pares que mejora/empeora/está estable y por cuánto (mediana de
`variacion_simetrica` condicionada al signo), y la matriz de transición
`pobre_t` x `pobre_th` (2x2) además de las 4 categorías de `transicion`.

**Corrección**: `pares_h4` incluye 28.343 pares con `identidad_confirmada ==
False` (probablemente otro hogar en la misma vivienda, no el mismo hogar) --
`par_valido` (Capa 2) los marca sin borrarlos. Todo lo de esta sección se
corre dos veces, sobre todos los pares y sobre `par_valido == True`, lado a
lado, para ver cuánto cambia.

In [ ]:
def tendencia_por_trimestre(pares: pd.DataFrame) -> pd.DataFrame:
    filas = []
    for (anio, trimestre), grupo in pares.groupby(["ANO4_t", "TRIMESTRE_t"]):
        filas.append({
            "ANO4_t": anio, "TRIMESTRE_t": trimestre, "n_pares": len(grupo),
            "mediana_variacion_simetrica": trayectoria_distancia.mediana_ponderada(
                grupo["variacion_simetrica"], grupo["PONDIH_t"]),
            "mediana_variacion_log": trayectoria_distancia.mediana_ponderada(
                grupo["variacion_log"], grupo["PONDIH_t"]),
            "pct_mejora": 100 * (grupo["direccion"] == "mejora").mean(),
            "pct_estable": 100 * (grupo["direccion"] == "estable").mean(),
            "pct_empeora": 100 * (grupo["direccion"] == "empeora").mean(),
            "mediana_variacion_si_mejora": grupo.loc[grupo["direccion"] == "mejora", "variacion_simetrica"].median(),
            "mediana_variacion_si_empeora": grupo.loc[grupo["direccion"] == "empeora", "variacion_simetrica"].median(),
        })
    return pd.DataFrame(filas).sort_values(["ANO4_t", "TRIMESTRE_t"]).reset_index(drop=True)


for h, pares in ((1, pares_h1), (4, pares_h4)):
    for etiqueta, subconjunto in (("todos", pares), ("par_valido", pares[pares["par_valido"]])):
        tendencia = tendencia_por_trimestre(subconjunto)
        tendencia.to_csv(META / f"trayectoria_tendencia_trimestre_h{h}_{etiqueta}.csv", index=False)
        print(f"h={h} ({etiqueta}): data/meta/trayectoria_tendencia_trimestre_h{h}_{etiqueta}.csv "
              f"({len(tendencia)} trimestres)")
        display(tendencia.describe().round(3))


In [ ]:
def matriz_transicion(pares: pd.DataFrame) -> pd.DataFrame:
    return pd.crosstab(pares["pobre_t"], pares["pobre_th"], dropna=False, margins=True)


for h, pares in ((1, pares_h1), (4, pares_h4)):
    for etiqueta, subconjunto in (("todos", pares), ("par_valido", pares[pares["par_valido"]])):
        matriz = matriz_transicion(subconjunto)
        matriz.to_csv(META / f"trayectoria_matriz_transicion_h{h}_{etiqueta}.csv")
        print(f"h={h} ({etiqueta}) -- matriz pobre_t x pobre_th:")
        display(matriz)
        print(f"h={h} ({etiqueta}) -- distribución de transicion (4 categorías):")
        display(100 * subconjunto["transicion"].value_counts(normalize=True, dropna=False).round(3))


## 2. Regresión a la media

Variación promedio por decil de `ratio_lp_t` (deciles sin ponderar, solo
descriptivo) -- línea base: si no hubiera ningún otro predictor, ¿cuánto
varía en promedio un hogar según dónde está hoy respecto de la línea?

In [ ]:
def regresion_a_la_media(pares: pd.DataFrame) -> pd.DataFrame:
    con_decil = pares.dropna(subset=["ratio_lp_t"]).copy()
    con_decil["decil_ratio_lp_t"] = pd.qcut(con_decil["ratio_lp_t"].astype(float), 10, labels=False, duplicates="drop") + 1
    return con_decil.groupby("decil_ratio_lp_t").agg(
        n_pares=("variacion_simetrica", "size"),
        ratio_lp_t_mediana=("ratio_lp_t", "median"),
        variacion_simetrica_media=("variacion_simetrica", "mean"),
        variacion_log_media=("variacion_log", "mean"),
    ).reset_index()


for h, pares in ((1, pares_h1), (4, pares_h4)):
    for etiqueta, subconjunto in (("todos", pares), ("par_valido", pares[pares["par_valido"]])):
        regresion = regresion_a_la_media(subconjunto)
        regresion.to_csv(META / f"trayectoria_regresion_media_decil_h{h}_{etiqueta}.csv", index=False)
        print(f"h={h} ({etiqueta}): data/meta/trayectoria_regresion_media_decil_h{h}_{etiqueta}.csv")
        display(regresion.round(3))


## 3. Match a `t+4` desagregado

**Ausencia estructural por rotación / vivienda que no reaparece**:
`cascada.clasificar_vinculacion_trimestre` (D24, reusado tal cual) separa,
por hogar en `t`, `vivienda_sin_par` (la vivienda ya no está en `t+4` --
mezcla rotación 2-2-2 teórica + atrición real, D9), `hogar_reemplazado` (la
vivienda sigue, ningún hogar matcheó exacto por `NRO_HOGAR`) y
`n_hogares_matcheados` (sí hay match exacto). **Hogar que no supera la
confirmación**: dentro de los matcheados, `identidad_confirmada` (D10, el/la
jefe/a de `t` reconocible en el núcleo de `t+4`) puede seguir dando
`False`/`NA`.

**Regla menos estricta, propuesta sin aplicar**: entre los matcheados con
`identidad_confirmada == False`, los que además tienen
`cambio_composicion == False` (D10: nadie entra ni sale del hogar, según el
emparejamiento por sexo+edad) son candidatos a "recuperarse" con una regla
que no exija específicamente al jefe/a sino a cualquier miembro del núcleo --
se cuenta cuántos son, no se cambia `panel.identidad_nucleo`.

In [ ]:
filas_vinculacion = []
for anio, trimestre in ORIGENES_H4:
    anio_th, trimestre_th = panel.sumar_trimestres(anio, trimestre, 4)
    hogar_t = pd.read_parquet(NUCLEO / "hogar" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet",
                               columns=["CODUSU", "NRO_HOGAR", "AGLOMERADO"])
    hogar_th = pd.read_parquet(NUCLEO / "hogar" / f"ANO4={anio_th}" / f"TRIMESTRE={trimestre_th}" / "data.parquet",
                                columns=["CODUSU", "NRO_HOGAR", "AGLOMERADO"])
    p_panel = panel_pares = PANEL_DIR / "pares_h4" / f"ANO4={anio}" / f"TRIMESTRE={trimestre}" / "data.parquet"
    if not p_panel.exists():
        continue
    pares_hogar = pd.read_parquet(p_panel)
    vinculacion = cascada.clasificar_vinculacion_trimestre(hogar_t, hogar_th, pares_hogar)
    vinculacion["ANO4_t"], vinculacion["TRIMESTRE_t"] = anio, trimestre
    filas_vinculacion.append(vinculacion)

vinculacion_h4 = pd.DataFrame(filas_vinculacion)
totales_vinculacion = vinculacion_h4[[
    "n_viviendas_t", "n_vinculada", "n_hogar_reemplazado", "n_vivienda_sin_par",
    "n_hogares_en_vivienda_sin_par", "n_hogares_en_vivienda_reemplazada",
    "n_hogares_sin_match_multihogar", "n_hogares_matcheados",
]].sum()
print("totales 2016T2-2024T4 (h=4):")
display(totales_vinculacion)


In [ ]:
n_confirmado = int((pares_h4["identidad_confirmada"] == True).sum())
n_no_confirmado = int((pares_h4["identidad_confirmada"] == False).sum())
n_ambiguo = int(pares_h4["identidad_confirmada"].isna().sum())
print(f"de {len(pares_h4):,} pares matcheados h=4: {n_confirmado:,} identidad_confirmada, "
      f"{n_no_confirmado:,} no confirmada, {n_ambiguo:,} jefe_ambiguo (NA)")

pares_h4_crudo = pd.concat([
    pd.read_parquet(PANEL_DIR / "pares_h4" / f"ANO4={a}" / f"TRIMESTRE={t}" / "data.parquet")
    for a, t in ORIGENES_H4
    if (PANEL_DIR / "pares_h4" / f"ANO4={a}" / f"TRIMESTRE={t}" / "data.parquet").exists()
], ignore_index=True)
no_confirmados = pares_h4_crudo[pares_h4_crudo["identidad_confirmada"] == False]
recuperables = int((no_confirmados["cambio_composicion"] == False).sum())
print(f"de {len(no_confirmados):,} pares no confirmados, {recuperables:,} "
      f"({100 * recuperables / len(no_confirmados):.1f}%) tienen cambio_composicion=False "
      f"-- candidatos a recuperarse con una regla menos estricta, no aplicada acá")

resumen_match_t4 = pd.DataFrame([{
    "n_identidad_confirmada": n_confirmado, "n_no_confirmada": n_no_confirmado,
    "n_jefe_ambiguo": n_ambiguo, "n_recuperables_regla_laxa": recuperables,
}])
resumen_match_t4.to_csv(META / "trayectoria_match_t4_desagregado.csv", index=False)
print("data/meta/trayectoria_match_t4_desagregado.csv escrito")


## 4. Conteos finales: train/holdout (split de D24/D32)

Holdout = origen `t >= 2023T1`; train = destino `t+h <= 2022T4`
(`panel.sumar_trimestres(2023, 1, -1)`); embargo de 1 trimestre entre medio.
Conteo clave: pares con `par_valido == True` por partición (la corrección
del punto 2 de Iván).

In [ ]:
HOLDOUT_DESDE = (2023, 1)
LIMITE_TRAIN_TH = panel.sumar_trimestres(*HOLDOUT_DESDE, -1)


def particion_train_holdout(pares: pd.DataFrame) -> pd.Series:
    es_holdout = pares["ANO4_t"].astype(int).gt(HOLDOUT_DESDE[0]) | (
        pares["ANO4_t"].astype(int).eq(HOLDOUT_DESDE[0]) & pares["TRIMESTRE_t"].astype(int).ge(HOLDOUT_DESDE[1]))
    es_train = pares["ANO4_th"].astype(int).lt(LIMITE_TRAIN_TH[0]) | (
        pares["ANO4_th"].astype(int).eq(LIMITE_TRAIN_TH[0]) & pares["TRIMESTRE_th"].astype(int).le(LIMITE_TRAIN_TH[1]))
    particion = pd.Series("embargo", index=pares.index, dtype="object")
    particion[es_train] = "train"
    particion[es_holdout] = "holdout"
    return particion


filas_conteo_final = []
for h, pares in ((1, pares_h1), (4, pares_h4)):
    for etiqueta, subconjunto in (("todos", pares), ("par_valido", pares[pares["par_valido"]])):
        particion = particion_train_holdout(subconjunto)
        for nombre_particion in ("train", "embargo", "holdout"):
            filas_conteo_final.append({"h": h, "subconjunto": etiqueta, "particion": nombre_particion,
                                        "n_pares": int((particion == nombre_particion).sum())})
conteos_finales = pd.DataFrame(filas_conteo_final)
conteos_finales.to_csv(META / "trayectoria_conteos_finales.csv", index=False)
print("todos los pares:")
display(conteos_finales[conteos_finales["subconjunto"] == "todos"].pivot(
    index="particion", columns="h", values="n_pares"))
print("solo par_valido == True (conteo clave):")
display(conteos_finales[conteos_finales["subconjunto"] == "par_valido"].pivot(
    index="particion", columns="h", values="n_pares"))


## 5. Subanálisis de Laura: salida de la pobreza entre los pobres

Universo: `pobre_t == True`. Reusa `transicion` de la Capa 2
(`"sale"`/`"sigue_pobre"`, umbral `ratio_lp_th >= 1`) para la tasa principal,
y agrega la sensibilidad de umbral (`1.0`/`1.1`/`1.2`) y la atrición (vía la
clasificación vivienda/hogar de la sección 3) sobre el mismo subconjunto.

In [ ]:
pobres_t_h4 = pares_h4[pares_h4["pobre_t"] == True].copy()
confirmados_pobres = pobres_t_h4[pobres_t_h4["identidad_confirmada"] == True]
determinados_pobres = confirmados_pobres.dropna(subset=["ratio_lp_th"])

print(f"hogares pobres en t con destino t+4 válido (pares_h4): {len(pobres_t_h4):,}")
print(f"de los cuales, identidad_confirmada: {len(confirmados_pobres):,}")
print(f"de los confirmados, ratio_lp_th no nulo: {len(determinados_pobres):,}")

filas_umbral = []
for umbral in (1.0, 1.1, 1.2):
    sale = determinados_pobres["ratio_lp_th"] >= umbral
    filas_umbral.append({"umbral": umbral, "n_total": len(determinados_pobres),
                          "n_sale": int(sale.sum()), "tasa_salida": sale.mean()})
sensibilidad_laura = pd.DataFrame(filas_umbral)
sensibilidad_laura.to_csv(META / "laura_salida_pobreza_sensibilidad_umbral.csv", index=False)
display(sensibilidad_laura)


In [ ]:
n_sin_ratio_th = len(confirmados_pobres) - len(determinados_pobres)
print(f"confirmados con ratio_lp_th nulo (no respuesta en t+4): {n_sin_ratio_th:,} "
      f"({100 * n_sin_ratio_th / len(confirmados_pobres):.1f}% de los confirmados)")

particion_laura = particion_train_holdout(determinados_pobres)
resumen_particion_laura = pd.DataFrame({"particion": particion_laura}).value_counts().rename("n_pares").reset_index()
resumen_particion_laura.to_csv(META / "laura_salida_pobreza_holdout.csv", index=False)
display(resumen_particion_laura)


---

## Cierre

Capa 3 completa: tendencia general, regresión a la media, desagregado de
match a `t+4` (con la regla menos estricta solo cuantificada, no aplicada) y
conteos finales train/holdout, todos sobre `data/05_trayectoria_distancia/`.
El subanálisis de Laura queda reescrito sobre la misma capa, sin reconstruir
ningún join a mano. Todas las salidas en `data/meta/trayectoria_*.csv` y
`data/meta/laura_*.csv`.